In [1]:
import numpy as np
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

## Load Data


In [2]:
# 2 past seasons + current
past_seasons = ["2024-25", "2025-26"]
current_season = "2026-27"
seasons = past_seasons + [current_season]

target_column = "total_points"
id_column = "element"
time_column = "GW"
random_state = 42

FPL_API = "https://fantasy.premierleague.com/api"
VAASTAV_GW = (
    "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/"
    "master/data/{season}/gws/merged_gw.csv"
)
VAASTAV_TEAMS = (
    "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/"
    "master/data/{season}/teams.csv"
)
VAASTAV_PLAYERS = (
    "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/"
    "master/data/{season}/players_raw.csv"
)
SEASON_ORDER = {"2024-25": 2024, "2025-26": 2025, "2026-27": 2026}
POSITION_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
K_BY_POSITION = {"GK": 5, "DEF": 10, "MID": 10, "FWD": 10}


In [3]:
def get_json(url):
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    response.raise_for_status()
    return response.json()


def load_past_season(season):
    data = pd.read_csv(VAASTAV_GW.format(season=season))
    data["season"] = season
    return data


def load_current_season(season):
    bootstrap = get_json(f"{FPL_API}/bootstrap-static/")

    teams = pd.DataFrame(bootstrap["teams"])[["id", "name"]].rename(
        columns={"id": "team_id", "name": "team"}
    )
    players = pd.DataFrame(bootstrap["elements"])[
        ["id", "web_name", "team", "element_type"]
    ].rename(columns={"id": "element", "team": "team_id", "web_name": "name"})
    players["position"] = players["element_type"].map(POSITION_MAP)
    players = players.merge(teams, on="team_id", how="left")

    def fetch_history(element_id):
        summary = get_json(f"{FPL_API}/element-summary/{element_id}/")
        history = pd.DataFrame(summary["history"])
        if history.empty:
            return history
        history["element"] = element_id
        return history

    histories = []
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {
            executor.submit(fetch_history, element_id): element_id
            for element_id in players["element"]
        }
        for future in as_completed(futures):
            history = future.result()
            if not history.empty:
                histories.append(history)

    data = pd.concat(histories, ignore_index=True)
    data = data.merge(
        players[["element", "name", "position", "team"]],
        on="element",
        how="left",
    )
    data["GW"] = data["round"]
    data["season"] = season
    data["xP"] = np.nan  # not in live element-summary
    return data, bootstrap


def load_team_lookup(bootstrap):
    frames = []
    for season in past_seasons:
        teams = pd.read_csv(VAASTAV_TEAMS.format(season=season))
        teams = teams[["id", "name"]].rename(
            columns={"id": "team_id", "name": "team_name"}
        )
        teams["season"] = season
        frames.append(teams)

    current = pd.DataFrame(bootstrap["teams"])[["id", "name"]].rename(
        columns={"id": "team_id", "name": "team_name"}
    )
    current["season"] = current_season
    frames.append(current)
    return pd.concat(frames, ignore_index=True)


def load_player_codes(bootstrap):
    frames = []
    for season in past_seasons:
        raw = pd.read_csv(VAASTAV_PLAYERS.format(season=season))
        part = raw[["id", "code"]].drop_duplicates("id")
        part = part.rename(columns={"id": "element"})
        part["season"] = season
        frames.append(part)

    current = pd.DataFrame(bootstrap["elements"])[["id", "code"]].rename(
        columns={"id": "element"}
    )
    current["season"] = current_season
    frames.append(current)
    return pd.concat(frames, ignore_index=True)


In [4]:
past_frames = [load_past_season(season) for season in past_seasons]
current_data, bootstrap = load_current_season(current_season)
team_lookup = load_team_lookup(bootstrap)
player_codes = load_player_codes(bootstrap)

data = pd.concat(past_frames + [current_data], ignore_index=True, sort=False)
data = data.merge(player_codes, on=["season", "element"], how="left")
data = data.sort_values(["season", time_column, id_column]).reset_index(drop=True)

print("Seasons:", seasons)
print("Shape:", data.shape)
print(data.groupby("season")[time_column].agg(["min", "max", "count"]))
display(data.head())
display(data.dtypes)


Seasons: ['2024-25', '2025-26', '2026-27']
Shape: (62582, 55)
         min  max  count
season                  
2024-25    1   38  27605
2025-26    1   38  29757
2026-27    1    8   5220


,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,value,was_home,yellow_cards,GW,season,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles,code
0,Fábio Ferreira Vieira,MID,Arsenal,2.3,0,0,0,0,0.0,1,...,55,True,0,1,2024-25,NaN,NaN,NaN,NaN,438098
1,Gabriel Fernando de Jesus,FWD,Arsenal,3.2,0,0,1,0,0.8,2,...,70,True,1,1,2024-25,NaN,NaN,NaN,NaN,205651
2,Gabriel dos Santos Magalhães,DEF,Arsenal,4.2,0,0,22,1,1.4,3,...,60,True,0,1,2024-25,NaN,NaN,NaN,NaN,226597
3,Kai Havertz,FWD,Arsenal,3.5,1,3,48,1,24.1,4,...,80,True,0,1,2024-25,NaN,NaN,NaN,NaN,219847
4,Karl Hein,GK,Arsenal,0.0,0,0,0,0,0.0,5,...,40,True,0,1,2024-25,NaN,NaN,NaN,NaN,463748


name                                object
position                            object
team                                object
xP                                 float64
assists                              int64
bonus                                int64
bps                                  int64
clean_sheets                         int64
creativity                          object
element                              int64
expected_assists                    object
expected_goal_involvements          object
expected_goals                      object
expected_goals_conceded             object
fixture                              int64
goals_conceded                       int64
goals_scored                         int64
ict_index                           object
influence                           object
kickoff_time                        object
minutes                              int64
mng_clean_sheets                   float64
mng_draw                           float64
mng_goals_s

## Basic EDA


Check on full `data` before split:
1) shape / seasons / gameweeks
2) categorical vs numeric columns
3) missing %
4) target (`total_points`) & position mix
5) double gameweeks (player-GW with >1 fixture row)


In [5]:
print("Shape:", data.shape)
print("Seasons:", sorted(data["season"].unique()))
print(data.groupby("season")[time_column].agg(["min", "max", "nunique", "count"]))

display(data.head())
display(data.dtypes)


Shape: (62582, 55)
Seasons: ['2024-25', '2025-26', '2026-27']
         min  max  nunique  count
season                           
2024-25    1   38       38  27605
2025-26    1   38       38  29757
2026-27    1    8        8   5220


,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,value,was_home,yellow_cards,GW,season,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles,code
0,Fábio Ferreira Vieira,MID,Arsenal,2.3,0,0,0,0,0.0,1,...,55,True,0,1,2024-25,NaN,NaN,NaN,NaN,438098
1,Gabriel Fernando de Jesus,FWD,Arsenal,3.2,0,0,1,0,0.8,2,...,70,True,1,1,2024-25,NaN,NaN,NaN,NaN,205651
2,Gabriel dos Santos Magalhães,DEF,Arsenal,4.2,0,0,22,1,1.4,3,...,60,True,0,1,2024-25,NaN,NaN,NaN,NaN,226597
3,Kai Havertz,FWD,Arsenal,3.5,1,3,48,1,24.1,4,...,80,True,0,1,2024-25,NaN,NaN,NaN,NaN,219847
4,Karl Hein,GK,Arsenal,0.0,0,0,0,0,0.0,5,...,40,True,0,1,2024-25,NaN,NaN,NaN,NaN,463748


name                                object
position                            object
team                                object
xP                                 float64
assists                              int64
bonus                                int64
bps                                  int64
clean_sheets                         int64
creativity                          object
element                              int64
expected_assists                    object
expected_goal_involvements          object
expected_goals                      object
expected_goals_conceded             object
fixture                              int64
goals_conceded                       int64
goals_scored                         int64
ict_index                           object
influence                           object
kickoff_time                        object
minutes                              int64
mng_clean_sheets                   float64
mng_draw                           float64
mng_goals_s

Check missing %


In [6]:
display(pd.DataFrame({
    "missing_rate": data.isna().mean(),
}).sort_values("missing_rate", ascending=False).head(20))

defensive_columns = [
    c for c in ["tackles", "clearances_blocks_interceptions", "recoveries"]
    if c in data.columns
]
print("Defensive-count missing rate by season:")
display(data.groupby("season")[defensive_columns].agg(lambda s: s.isna().mean()))


,missing_rate
mng_win,0.785449
mng_goals_scored,0.785449
mng_clean_sheets,0.785449
mng_draw,0.785449
mng_underdog_win,0.785449
mng_underdog_draw,0.785449
mng_loss,0.785449
tackles,0.441101
recoveries,0.441101
defensive_contribution,0.441101


Defensive-count missing rate by season:


,tackles,clearances_blocks_interceptions,recoveries
season,,,
2024-25,1.0,1.0,1.0
2025-26,0.0,0.0,0.0
2026-27,0.0,0.0,0.0


Check categorical / non-numeric columns


In [7]:
non_numeric_columns = data.select_dtypes(exclude=["number", "bool"]).columns.tolist()
print("Non-numeric columns:", non_numeric_columns)

for column in ["position", "team", "season"]:
    if column in data.columns:
        print(f"\n{column}:")
        display(data[column].value_counts(dropna=False))


Non-numeric columns: ['name', 'position', 'team', 'creativity', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'ict_index', 'influence', 'kickoff_time', 'threat', 'season']

position:


position
MID    27884
DEF    20585
FWD     6923
GK      6868
AM       322
Name: count, dtype: int64


team:


team
Brighton          3795
Spurs             3511
Aston Villa       3508
Chelsea           3305
Man Utd           3230
Bournemouth       3198
Wolves            3102
Crystal Palace    3102
Brentford         3083
Newcastle         2968
Nott'm Forest     2951
Liverpool         2933
Man City          2904
Arsenal           2852
Everton           2758
West Ham          2569
Fulham            2505
Sunderland        2043
Burnley           1619
Leeds             1569
Southampton       1484
Leicester         1347
Ipswich           1339
Hull City          333
Coventry City      289
Ipswich Town       285
Name: count, dtype: int64


season:


season
2025-26    29757
2024-25    27605
2026-27     5220
Name: count, dtype: int64

Target distribution (`total_points`) by position


In [8]:
display(data[target_column].describe())
display(
    data.groupby("position")[target_column].describe()
)

# zero-point share (DNP / blanks)
zero_rate = (
    data.assign(is_zero=data[target_column] == 0)
    .groupby("position")["is_zero"]
    .mean()
    .rename("zero_points_rate")
)
display(zero_rate)


count    62582.000000
mean         1.189144
std          2.393278
min         -5.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         25.000000
Name: total_points, dtype: float64

,count,mean,std,min,25%,50%,75%,max
position,,,,,,,,
AM,322.0,5.779503,5.218988,0.0,1.0,5.0,9.0,21.0
DEF,20585.0,1.150838,2.357735,-5.0,0.0,0.0,1.0,24.0
FWD,6923.0,1.312870,2.614838,-2.0,0.0,0.0,1.0,19.0
GK,6868.0,0.820763,1.982021,-3.0,0.0,0.0,0.0,16.0
MID,27884.0,1.224430,2.343166,-2.0,0.0,0.0,2.0,25.0


position
AM     0.220497
DEF    0.625650
FWD    0.582840
GK     0.761939
MID    0.575814
Name: zero_points_rate, dtype: float64

Double gameweeks: >1 fixture row per player-GW


In [9]:
player_gw_counts = (
    data.groupby(["season", id_column, time_column])
    .size()
    .rename("n_fixture_rows")
    .reset_index()
)

print("Rows per player-GW:")
display(player_gw_counts["n_fixture_rows"].value_counts().sort_index())

dgw = player_gw_counts[player_gw_counts["n_fixture_rows"] > 1]
print("DGW player-GW count:", len(dgw))
display(dgw.head())


Rows per player-GW:


n_fixture_rows
1    60996
2      793
Name: count, dtype: int64

DGW player-GW count: 793


,season,element,GW,n_fixture_rows
32,2024-25,1,33,2
69,2024-25,2,33,2
106,2024-25,3,33,2
143,2024-25,4,33,2
180,2024-25,5,33,2


## Next gameweek


In [10]:
# The choice loop uses 2025-26 GW20-38. This cell only finds the finished
# weeks and builds the next gameweek, which has no points yet.
events = pd.DataFrame(bootstrap["events"])[
    ["id", "name", "finished", "is_current", "is_next"]
].rename(columns={"id": "GW"})

# finished = GWs we actually have labels for in current `data`
# (avoids holes when element-summary is ahead/behind bootstrap flags)
finished_gws = (
    data.loc[
        (data["season"] == current_season) & data[target_column].notna(),
        time_column,
    ]
    .drop_duplicates()
    .sort_values()
    .astype(int)
    .tolist()
)

next_gw = events.loc[events["is_next"], "GW"]
test_gw = int(next_gw.iloc[0]) if len(next_gw) else (finished_gws[-1] + 1)

finished_gws = [gw for gw in finished_gws if gw != test_gw]

# upcoming GW may not be in `data` yet (no labels) -> build from fixtures + bootstrap
fixtures = pd.DataFrame(get_json(f"{FPL_API}/fixtures/"))
fixtures = fixtures[fixtures["event"] == test_gw][
    ["id", "event", "team_h", "team_a"]
].rename(columns={"id": "fixture", "event": "GW"})

players = pd.DataFrame(bootstrap["elements"])[
    ["id", "web_name", "team", "element_type", "now_cost"]
].rename(
    columns={
        "id": "element",
        "team": "team_id",
        "web_name": "name",
        "now_cost": "value",
    }
)
teams = pd.DataFrame(bootstrap["teams"])[["id", "name"]].rename(
    columns={"id": "team_id", "name": "team"}
)
players["position"] = players["element_type"].map(POSITION_MAP)
players = players.merge(teams, on="team_id", how="left")

home = fixtures[["fixture", "GW", "team_h", "team_a"]].rename(
    columns={"team_h": "team_id", "team_a": "opponent_team"}
)
home["was_home"] = True
away = fixtures[["fixture", "GW", "team_a", "team_h"]].rename(
    columns={"team_a": "team_id", "team_h": "opponent_team"}
)
away["was_home"] = False
fixture_rows = pd.concat([home, away], ignore_index=True)

test_df = players.merge(fixture_rows, on="team_id", how="inner")
test_df["season"] = current_season
test_df[target_column] = np.nan
test_df = test_df.merge(
    player_codes[["season", "element", "code"]],
    on=["season", "element"],
    how="left",
)

print("Finished GWs (labeled in data):", finished_gws)
print("Test GW:", test_gw)
print("test_df:", test_df.shape, "GW:", sorted(test_df[time_column].unique()))
display(test_df.head())


Finished GWs (labeled in data): [1, 2, 3, 4, 5, 6, 7, 8]
Test GW: 9
test_df: (669, 14) GW: [np.int64(9)]


,element,name,team_id,element_type,value,position,team,fixture,GW,opponent_team,was_home,season,total_points,code
0,1,Raya,1,1,62,GK,Arsenal,87,9,14,False,2026-27,NaN,154561
1,2,Arrizabalaga,1,1,49,GK,Arsenal,87,9,14,False,2026-27,NaN,109745
2,3,Meslier,1,1,49,GK,Arsenal,87,9,14,False,2026-27,NaN,437495
3,4,Gabriel,1,2,79,DEF,Arsenal,87,9,14,False,2026-27,NaN,226597
4,5,J.Timber,1,2,65,DEF,Arsenal,87,9,14,False,2026-27,NaN,445122


## Feature Engineering


Same `feature_engineer` for every row. A gameweek never sees its own result.

History follows the player `code` across seasons. Team attack and defense rates do the same: the previous five matches can start in the season before. Double gameweeks keep a count of home matches, and they keep the harder opponent band and the easier opponent band.

Team bands use league position only. Past season is the official final table. Current season is the table before that gameweek, and only after the team has already played five matches. Before that, the current band stays at last season's band. A club that was not in the league last season gets past band 0.

Defensive-action counts are missing for 2024-25, so those rows stay missing. Recent minutes is a column. It does not also weight the fit. The fit uses `add_top_group_target`.

The frozen model keeps its original columns. A shadow challenger uses position-specific columns: recent scoring and creation, the defensive-contribution threshold rate, bonus/BPS history, and the team's home/away matchup. Venue rates use only the previous five matches in the same home/away role. For example, an away forward gets his team's away attack against the opponent's home defence.



In [ ]:
LEAGUE_EURO_PLACES = {
    # cl, europa, conference from league position only. Cups are ignored,
    # so the conference code is unused on purpose.
    "2023-24": (4, 1, 0),
    "2024-25": (5, 1, 0),
    "2025-26": (5, 1, 0),
    # This season has not earned an extra Champions League place.
    "2026-27": (4, 1, 0),
}

# Official final positions. Names match the vaastav / bootstrap team strings.
FINAL_POSITIONS = {
    "2023-24": {
        "Man City": 1, "Arsenal": 2, "Liverpool": 3, "Aston Villa": 4, "Spurs": 5,
        "Chelsea": 6, "Newcastle": 7, "Man Utd": 8, "West Ham": 9, "Crystal Palace": 10,
        "Brighton": 11, "Bournemouth": 12, "Fulham": 13, "Wolves": 14, "Everton": 15,
        "Brentford": 16, "Nott'm Forest": 17, "Luton": 18, "Burnley": 19, "Sheffield Utd": 20,
    },
    "2024-25": {
        "Liverpool": 1, "Arsenal": 2, "Man City": 3, "Chelsea": 4, "Newcastle": 5,
        "Aston Villa": 6, "Nott'm Forest": 7, "Brighton": 8, "Bournemouth": 9, "Brentford": 10,
        "Fulham": 11, "Crystal Palace": 12, "Everton": 13, "West Ham": 14, "Man Utd": 15,
        "Wolves": 16, "Spurs": 17, "Leicester": 18, "Ipswich": 19, "Southampton": 20,
    },
    "2025-26": {
        "Arsenal": 1, "Man City": 2, "Man Utd": 3, "Aston Villa": 4, "Liverpool": 5,
        "Bournemouth": 6, "Sunderland": 7, "Brighton": 8, "Brentford": 9, "Chelsea": 10,
        "Fulham": 11, "Newcastle": 12, "Everton": 13, "Leeds": 14, "Crystal Palace": 15,
        "Nott'm Forest": 16, "Spurs": 17, "West Ham": 18, "Burnley": 19, "Wolves": 20,
    },
}

PREVIOUS_SEASON = {
    "2024-25": "2023-24",
    "2025-26": "2024-25",
    "2026-27": "2025-26",
}


def band_from_rank(rank, season):
    """7 is Champions League, 0 is not used here. 1 is the bottom of the league."""
    if pd.isna(rank):
        return np.nan
    rank = int(rank)
    champions_league, europa, conference = LEAGUE_EURO_PLACES[season]
    if rank <= champions_league:
        return 7
    if rank <= champions_league + europa:
        return 6
    if rank <= champions_league + europa + conference:
        return 5
    if rank <= 10:
        return 4
    if rank <= 14:
        return 3
    if rank <= 17:
        return 2
    return 1


def past_band_for(team, season):
    """Final band from the previous season. 0 means the club was not in the league."""
    previous = PREVIOUS_SEASON.get(season)
    if previous is None:
        return 0
    rank = FINAL_POSITIONS[previous].get(team)
    if rank is None:
        return 0
    return band_from_rank(rank, previous)


def ranks_from_stats(stats):
    """Points, then goal difference, then goals scored, then name."""
    played = [
        (team, item["pts"], item["gd"], item["gf"])
        for team, item in stats.items()
        if item["played"] > 0
    ]
    played.sort(key=lambda row: (-row[1], -row[2], -row[3], row[0]))
    return {team: rank for rank, (team, *_) in enumerate(played, start=1)}


def build_current_bands(data):
    """Band from this season's table using only matches before that gameweek."""
    scored = data.dropna(subset=["team_h_score", "team_a_score", "team"])
    home = scored.loc[scored["was_home"] == True, [
        "season", time_column, "fixture", "team", "team_h_score", "team_a_score",
    ]]
    home = home.drop_duplicates(["season", "fixture", "team"])
    home = home.rename(columns={"team_h_score": "gf", "team_a_score": "ga"})
    away = scored.loc[scored["was_home"] == False, [
        "season", time_column, "fixture", "team", "team_h_score", "team_a_score",
    ]]
    away = away.drop_duplicates(["season", "fixture", "team"])
    away = away.rename(columns={"team_a_score": "gf", "team_h_score": "ga"})
    matches = pd.concat([home, away], ignore_index=True)
    matches["pts"] = np.where(
        matches["gf"] > matches["ga"], 3, np.where(matches["gf"] == matches["ga"], 1, 0)
    )
    matches["gd"] = matches["gf"] - matches["ga"]

    team_gws = data[["season", time_column, "team"]].dropna().drop_duplicates()
    rows = []
    for season, season_teams in team_gws.groupby("season"):
        teams = sorted(season_teams["team"].unique())
        stats = {team: {"pts": 0, "gd": 0, "gf": 0, "played": 0} for team in teams}
        season_matches = matches[matches["season"] == season]
        for gw in sorted(season_teams[time_column].unique()):
            ranks = ranks_from_stats(stats)
            for team in teams:
                rank = ranks.get(team)
                played_before = stats[team]["played"]
                band = (
                    band_from_rank(rank, season)
                    if rank is not None and played_before >= 5
                    else np.nan
                )
                rows.append({
                    "season": season,
                    time_column: gw,
                    "team": team,
                    "team_current_band": band,
                })
            played_now = season_matches[season_matches[time_column] == gw]
            for record in played_now.itertuples(index=False):
                item = stats[record.team]
                item["pts"] += record.pts
                item["gd"] += record.gd
                item["gf"] += record.gf
                item["played"] += 1
    return pd.DataFrame(rows)


def build_team_gw(data):
    """Team goals and goals-against per gameweek, from that gameweek's matches."""
    match_xg = (
        data.groupby(["season", time_column, "fixture", "team"], as_index=False)[
            "expected_goals"
        ]
        .sum()
        .rename(columns={"expected_goals": "match_xg"})
    )
    pairs = data.groupby(
        ["season", time_column, "fixture", "team", "opponent_name"], as_index=False
    ).size()
    pairs = pairs.drop(columns="size")
    pairs = pairs.merge(
        match_xg, on=["season", time_column, "fixture", "team"], how="left"
    )
    opponent_xg = match_xg.rename(
        columns={"team": "opponent_name", "match_xg": "opp_xg"}
    )
    pairs = pairs.merge(
        opponent_xg,
        on=["season", time_column, "fixture", "opponent_name"],
        how="left",
    )
    return pairs.groupby(["season", time_column, "team"], as_index=False).agg(
        team_xg=("match_xg", "sum"),
        team_xga=("opp_xg", "sum"),
        team_n_fixtures=("fixture", "nunique"),
    )


def add_team_rates(team_gw):
    """Previous 5 matches. The window continues from the previous season."""
    team_gw = team_gw.copy()
    team_gw["season_order"] = team_gw["season"].map(SEASON_ORDER)
    team_gw = team_gw.sort_values(["team", "season_order", time_column]).reset_index(drop=True)
    grouped = team_gw.groupby("team", sort=False)

    def previous_rate(value_column):
        numerator = grouped[value_column].transform(
            lambda s: s.shift(1).rolling(5, min_periods=1).sum()
        )
        denominator = grouped["team_n_fixtures"].transform(
            lambda s: s.shift(1).rolling(5, min_periods=1).sum()
        )
        return numerator / denominator.replace(0, np.nan)

    team_gw["team_xg_per_match_5"] = previous_rate("team_xg")
    team_gw["team_xga_per_match_5"] = previous_rate("team_xga")

    league = team_gw.groupby(["season", time_column])[
        ["team_xg_per_match_5", "team_xga_per_match_5"]
    ].transform("mean")
    team_gw["xg_rate"] = team_gw["team_xg_per_match_5"] / league["team_xg_per_match_5"]
    team_gw["xga_rate"] = team_gw["team_xga_per_match_5"] / league["team_xga_per_match_5"]
    return team_gw.drop(columns="season_order")


def player_opponent_rates(data, team_gw):
    """Average the upcoming opponents' pre-gameweek rates. Double gameweeks are averaged."""
    rates = team_gw.rename(
        columns={
            "team": "opponent_name",
            "xg_rate": "opp_xg_rate",
            "xga_rate": "opp_xga_rate",
        }
    )
    long = data[
        ["season", id_column, time_column, "fixture", "opponent_name"]
    ].drop_duplicates()
    long = long.merge(
        rates[["season", time_column, "opponent_name", "opp_xg_rate", "opp_xga_rate"]],
        on=["season", time_column, "opponent_name"],
        how="left",
    )
    return long.groupby(["season", id_column, time_column], as_index=False)[
        ["opp_xg_rate", "opp_xga_rate"]
    ].mean()


def build_team_fixture(data):
    """One row per team-fixture with xG, xGA, venue, and the actual clean-sheet result."""
    data = data.copy()
    data["expected_goals"] = pd.to_numeric(data["expected_goals"], errors="coerce")
    match_xg = (
        data.groupby(["season", time_column, "fixture", "team"], as_index=False)[
            "expected_goals"
        ]
        .sum(min_count=1)
        .rename(columns={"expected_goals": "team_xg"})
    )
    fixture_team = data[[
        "season", time_column, "fixture", "team", "opponent_name", "was_home",
        "team_h_score", "team_a_score",
    ]].drop_duplicates(["season", "fixture", "team"])
    fixture_team = fixture_team.merge(
        match_xg, on=["season", time_column, "fixture", "team"], how="left"
    )
    opponent_xg = match_xg.rename(
        columns={"team": "opponent_name", "team_xg": "team_xga"}
    )
    fixture_team = fixture_team.merge(
        opponent_xg,
        on=["season", time_column, "fixture", "opponent_name"],
        how="left",
    )
    fixture_team["goals_against"] = np.where(
        fixture_team["was_home"],
        fixture_team["team_a_score"],
        fixture_team["team_h_score"],
    )
    fixture_team["team_clean_sheet"] = np.where(
        fixture_team["goals_against"].notna(),
        (fixture_team["goals_against"] == 0).astype(float),
        np.nan,
    )
    return fixture_team


def add_team_venue_rates(team_fixture):
    """Previous five same-venue matches. Fixtures in one GW share the same prior history."""
    team_fixture = team_fixture.copy()
    team_fixture["season_order"] = team_fixture["season"].map(SEASON_ORDER)
    team_fixture = team_fixture.sort_values(
        ["team", "was_home", "season_order", time_column, "fixture"]
    ).reset_index(drop=True)
    rate_columns = {
        "team_xg": "team_xg_venue_5",
        "team_xga": "team_xga_venue_5",
        "team_clean_sheet": "team_clean_sheet_venue_rate_5",
    }
    for target in rate_columns.values():
        team_fixture[target] = np.nan

    for _, group in team_fixture.groupby(["team", "was_home"], sort=False):
        history = {source: [] for source in rate_columns}
        for _, gw_part in group.groupby(["season_order", time_column], sort=False):
            for source, target in rate_columns.items():
                values = history[source][-5:]
                if values:
                    team_fixture.loc[gw_part.index, target] = float(np.mean(values))
            for source in rate_columns:
                current = gw_part[source].dropna().astype(float).tolist()
                history[source].extend(current)

    return team_fixture.drop(columns="season_order")


def player_venue_matchups(data, team_fixture):
    """Pre-game venue attack and defence for each player-fixture, then one player-GW row."""
    keys = ["season", time_column, "fixture"]
    own_columns = [
        "team_xg_venue_5", "team_xga_venue_5", "team_clean_sheet_venue_rate_5",
    ]
    fixture_rows = data[
        keys + [id_column, "team", "opponent_name"]
    ].drop_duplicates()
    fixture_rows = fixture_rows.merge(
        team_fixture[keys + ["team"] + own_columns],
        on=keys + ["team"],
        how="left",
    )
    opponent = team_fixture[keys + ["team", "team_xg_venue_5", "team_xga_venue_5"]].rename(
        columns={
            "team": "opponent_name",
            "team_xg_venue_5": "opp_xg_venue_5",
            "team_xga_venue_5": "opp_xga_venue_5",
        }
    )
    fixture_rows = fixture_rows.merge(
        opponent, on=keys + ["opponent_name"], how="left"
    )
    fixture_rows["venue_attack_matchup"] = (
        fixture_rows["team_xg_venue_5"] * fixture_rows["opp_xga_venue_5"]
    )
    fixture_rows["venue_defense_matchup"] = (
        fixture_rows["team_xga_venue_5"] * fixture_rows["opp_xg_venue_5"]
    )
    value_columns = own_columns + [
        "opp_xg_venue_5", "opp_xga_venue_5",
        "venue_attack_matchup", "venue_defense_matchup",
    ]
    return fixture_rows.groupby(
        ["season", id_column, time_column], as_index=False
    )[value_columns].mean()


def feature_engineer(data, beta=0.9):
    """X-side only. History uses earlier rows of the same player, including past seasons."""
    data = data.copy()

    drop_columns = [c for c in data.columns if c.startswith("mng_")]
    data = data.drop(columns=drop_columns, errors="ignore")

    numeric_object_columns = [
        "creativity", "influence", "threat", "ict_index",
        "expected_goals", "expected_assists", "expected_goal_involvements",
        "expected_goals_conceded", "xP",
    ]
    for column in numeric_object_columns:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

    if "position" in data.columns:
        data["position"] = data["position"].replace({"AM": "MID"})

    data["season_order"] = data["season"].map(SEASON_ORDER)
    data["player_key"] = np.where(
        data["code"].notna(),
        "code-" + data["code"].astype("Int64").astype(str),
        data["season"].astype(str) + "-" + data[id_column].astype(str),
    )

    lookup = team_lookup.rename(
        columns={"team_id": "opponent_team", "team_name": "opponent_name"}
    )
    data = data.drop(columns=["opponent_name"], errors="ignore")
    data = data.merge(
        lookup[["season", "opponent_team", "opponent_name"]],
        on=["season", "opponent_team"],
        how="left",
    )

    band_table = build_current_bands(data)
    band_table["team_past_band"] = [
        past_band_for(team, season)
        for team, season in zip(band_table["team"], band_table["season"])
    ]
    band_table["team_current_band"] = band_table["team_current_band"].fillna(
        band_table["team_past_band"]
    )

    opponent_bands = band_table.rename(
        columns={
            "team": "opponent_name",
            "team_past_band": "opp_past_band",
            "team_current_band": "opp_current_band",
        }
    )
    long_opponents = data[
        ["season", id_column, time_column, "fixture", "opponent_name"]
    ].drop_duplicates()
    long_opponents = long_opponents.merge(
        opponent_bands[
            ["season", time_column, "opponent_name", "opp_past_band", "opp_current_band"]
        ],
        on=["season", time_column, "opponent_name"],
        how="left",
    )
    # a double keeps the harder band and the easier band
    opponent_band_span = long_opponents.groupby(
        ["season", id_column, time_column], as_index=False
    ).agg(
        opp_past_band_hard=("opp_past_band", "max"),
        opp_past_band_easy=("opp_past_band", "min"),
        opp_current_band_hard=("opp_current_band", "max"),
        opp_current_band_easy=("opp_current_band", "min"),
    )

    team_gw = add_team_rates(build_team_gw(data))
    opponent_rates = player_opponent_rates(data, team_gw)
    team_fixture = add_team_venue_rates(build_team_fixture(data))
    venue_matchups = player_venue_matchups(data, team_fixture)

    # Fixture-level threshold events are calculated before double gameweeks collapse.
    data["n_fixtures"] = 1
    data["n_home"] = data["was_home"].fillna(False).astype(int)
    defender_parts = [
        c for c in ["tackles", "clearances_blocks_interceptions"] if c in data.columns
    ]
    outfield_parts = [
        c for c in ["tackles", "clearances_blocks_interceptions", "recoveries"]
        if c in data.columns
    ]
    defender_count = data[defender_parts].sum(axis=1, min_count=1)
    outfield_count = data[outfield_parts].sum(axis=1, min_count=1)
    fixture_def_count = pd.Series(
        np.where(data["position"] == "DEF", defender_count, outfield_count),
        index=data.index,
        dtype=float,
    )
    def_threshold = data["position"].map({"DEF": 10, "MID": 12, "FWD": 12})
    data["def_contribution_hit"] = np.nan
    known_def = fixture_def_count.notna() & def_threshold.notna()
    data.loc[known_def, "def_contribution_hit"] = (
        fixture_def_count.loc[known_def] >= def_threshold.loc[known_def]
    ).astype(float)
    data["bonus_hit"] = np.where(
        data["bonus"].notna(), (data["bonus"] > 0).astype(float), np.nan
    )
    data["eligible_clean_sheet_hit"] = np.where(
        data["clean_sheets"].notna() & data["minutes"].notna(),
        ((data["clean_sheets"] > 0) & (data["minutes"] >= 60)).astype(float),
        np.nan,
    )

    # one row per player-gameweek
    group_keys = ["season", id_column, time_column]
    sum_columns = [
        c
        for c in [
            "minutes", "total_points", "assists", "bonus", "bps", "clean_sheets",
            "goals_conceded", "goals_scored", "own_goals", "penalties_missed",
            "penalties_saved", "saves", "yellow_cards", "red_cards", "starts",
            "expected_goals", "expected_assists", "expected_goal_involvements",
            "expected_goals_conceded", "influence", "creativity", "threat", "ict_index",
            "clearances_blocks_interceptions", "recoveries", "tackles",
            "defensive_contribution", "def_contribution_hit", "bonus_hit",
            "eligible_clean_sheet_hit", "transfers_in", "transfers_out",
            "transfers_balance", "selected", "n_fixtures", "n_home",
        ]
        if c in data.columns
    ]
    first_columns = [
        c
        for c in [
            "name", "position", "team", "value", "code", "player_key", "season_order",
        ]
        if c in data.columns
    ]
    def sum_keep_missing(series):
        return series.sum(min_count=1)

    agg = {c: sum_keep_missing for c in sum_columns}
    agg.update({c: "first" for c in first_columns})
    data = data.groupby(group_keys, as_index=False).agg(agg)

    data = data.merge(
        band_table[
            ["season", time_column, "team", "team_past_band", "team_current_band"]
        ],
        on=["season", time_column, "team"],
        how="left",
    )
    data = data.merge(
        opponent_band_span, on=["season", id_column, time_column], how="left"
    )

    # player history across seasons, past rows only
    data = data.sort_values(["player_key", "season_order", time_column]).reset_index(drop=True)
    player_group = data.groupby("player_key", sort=False)

    data["minutes_ewm"] = player_group["minutes"].transform(
        lambda s: s.ewm(alpha=1 - beta, adjust=True).mean().shift(1)
    )
    data["zero_min_rate_5"] = player_group["minutes"].transform(
        lambda s: (s == 0).astype(float).shift(1).rolling(5, min_periods=1).mean()
    )
    start_share = data["starts"].fillna(0) / data["n_fixtures"]
    data["start_rate_5"] = start_share.groupby(data["player_key"], sort=False).transform(
        lambda s: s.shift(1).rolling(5, min_periods=1).mean()
    )
    full_share = data["minutes"].fillna(0) / data["n_fixtures"]
    data["minutes_60_rate"] = (full_share >= 60).astype(float).groupby(
        data["player_key"], sort=False
    ).transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

    def history_per90(column, skip_missing=False):
        value = pd.to_numeric(data[column], errors="coerce")
        minutes = data["minutes"].fillna(0.0)
        if skip_missing:
            minutes = minutes.where(value.notna(), 0.0)
        value = value.fillna(0.0)
        temp = pd.DataFrame({"player_key": data["player_key"], "_value": value, "_minutes": minutes})
        grouped = temp.groupby("player_key", sort=False)
        cum_value = grouped["_value"].transform(lambda s: s.cumsum().shift(1))
        cum_minutes = grouped["_minutes"].transform(lambda s: s.cumsum().shift(1))
        return np.where(cum_minutes > 0, cum_value / cum_minutes * 90.0, np.nan)

    def recent_per90(column, window):
        value = pd.to_numeric(data[column], errors="coerce").fillna(0.0)
        minutes = data["minutes"].fillna(0.0)
        temp = pd.DataFrame({"player_key": data["player_key"], "_value": value, "_minutes": minutes})
        grouped = temp.groupby("player_key", sort=False)
        roll_value = grouped["_value"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).sum()
        )
        roll_minutes = grouped["_minutes"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).sum()
        )
        return np.where(roll_minutes > 0, roll_value / roll_minutes * 90.0, np.nan)

    data["xg_per90"] = history_per90("expected_goals") if "expected_goals" in data.columns else np.nan
    data["xa_per90"] = history_per90("expected_assists") if "expected_assists" in data.columns else np.nan
    data["saves_per90"] = history_per90("saves") if "saves" in data.columns else np.nan
    data["xg_per90_5"] = recent_per90("expected_goals", 5)
    data["xa_per90_5"] = recent_per90("expected_assists", 5)
    data["xg_per90_10"] = recent_per90("expected_goals", 10)
    data["xa_per90_10"] = recent_per90("expected_assists", 10)
    data["goals_per90_10"] = recent_per90("goals_scored", 10)
    data["assists_per90_10"] = recent_per90("assists", 10)
    data["threat_per90_5"] = recent_per90("threat", 5)
    data["creativity_per90_5"] = recent_per90("creativity", 5)
    data["saves_per90_5"] = recent_per90("saves", 5)
    data["bps_per90_5"] = recent_per90("bps", 5)
    data["xg_form"] = data["xg_per90_5"] - data["xg_per90"]
    data["xa_form"] = data["xa_per90_5"] - data["xa_per90"]

    bonus_share = data["bonus_hit"] / data["n_fixtures"].replace(0, np.nan)
    data["bonus_rate_5"] = bonus_share.groupby(
        data["player_key"], sort=False
    ).transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    clean_sheet_share = (
        data["eligible_clean_sheet_hit"] / data["n_fixtures"].replace(0, np.nan)
    )
    data["clean_sheet_rate_5"] = clean_sheet_share.groupby(
        data["player_key"], sort=False
    ).transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

    def_parts = [
        c for c in ["tackles", "clearances_blocks_interceptions", "recoveries"]
        if c in data.columns
    ]
    if def_parts:
        # A season with no defensive counts stays missing. Do not treat it as zero.
        data["def_actions"] = data[def_parts].sum(axis=1, min_count=1)
        data["def_actions_per90"] = history_per90("def_actions", skip_missing=True)
    else:
        data["def_actions_per90"] = np.nan

    def_contribution_share = (
        data["def_contribution_hit"] / data["n_fixtures"].replace(0, np.nan)
    )
    data["def_contribution_rate_5"] = def_contribution_share.groupby(
        data["player_key"], sort=False
    ).transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

    # Legacy role ratio. The shadow challenger uses the team attack ratio below.
    attack = data["xg_per90"] + data["xa_per90"]
    denom = attack + data["def_actions_per90"]
    data["attacking_share"] = np.where(
        data["xg_per90"].notna()
        & data["xa_per90"].notna()
        & data["def_actions_per90"].notna()
        & (denom > 0),
        attack / denom,
        np.nan,
    )

    data["minutes_avg_3"] = player_group["minutes"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
    data["minutes_avg_10"] = player_group["minutes"].transform(
        lambda s: s.shift(1).rolling(10, min_periods=1).mean()
    )
    data["minutes_form"] = data["minutes_avg_3"] - data["minutes_avg_10"]

    data = data.merge(
        team_gw[[
            "season", time_column, "team",
            "team_xg_per_match_5", "team_xga_per_match_5",
        ]],
        on=["season", time_column, "team"],
        how="left",
    )
    data = data.merge(opponent_rates, on=["season", id_column, time_column], how="left")
    data = data.merge(
        venue_matchups, on=["season", id_column, time_column], how="left"
    )
    data["xg_matchup"] = data["xg_per90"] * data["opp_xga_rate"]
    data["xa_matchup"] = data["xa_per90"] * data["opp_xga_rate"]
    data["concede_matchup"] = data["team_xga_per_match_5"] * data["opp_xg_rate"]
    data["xgi_team_rate_ratio_5"] = (
        (data["xg_per90_5"] + data["xa_per90_5"])
        / data["team_xg_per_match_5"].replace(0, np.nan)
    )

    return data


def add_top_group_target(data, points_column="total_points"):
    """1 when the tie-aware rank is inside that position's K, else 0."""
    data = data.copy()
    data["y"] = np.nan
    labeled = data[points_column].notna()
    rank = (
        data.loc[labeled]
        .groupby(["season", time_column, "position"])[points_column]
        .rank(method="min", ascending=False)
    )
    k = data.loc[labeled, "position"].map(K_BY_POSITION)
    data.loc[labeled, "y"] = (rank.to_numpy() <= k.to_numpy()).astype(float)
    return data


In [ ]:
# engineer on the finished panel plus the next gameweek, so that week gets real history
test_stub = test_df.copy()
for column in data.columns:
    if column not in test_stub.columns:
        test_stub[column] = np.nan

# unknown outcomes for upcoming GW
for column in [
    "minutes",
    "total_points",
    "assists",
    "bonus",
    "bps",
    "clean_sheets",
    "goals_conceded",
    "goals_scored",
    "saves",
    "starts",
    "expected_goals",
    "expected_assists",
]:
    if column in test_stub.columns:
        test_stub[column] = 0 if column != "total_points" else np.nan

# if test_gw already finished and present in data, do not duplicate it
if ((data["season"] == current_season) & (data[time_column] == test_gw)).any():
    panel = data.copy()
else:
    panel = pd.concat([data, test_stub], ignore_index=True, sort=False)

panel = feature_engineer(panel)

test_df = panel[
    (panel["season"] == current_season)
    & (panel[time_column] == test_gw)
].copy()

print("test_df:", test_df.shape)

labeled_preview = panel[panel[target_column].notna()].copy()
labeled_preview = add_top_group_target(labeled_preview)
print("Share of player-weeks in the top group:")
display(labeled_preview.groupby("position")["y"].mean().reindex(list(K_BY_POSITION)))
group_size = (
    labeled_preview.groupby(["season", time_column, "position"], as_index=False)["y"]
    .sum()
    .rename(columns={"y": "n_true"})
)
group_size["k"] = group_size["position"].map(K_BY_POSITION)
print("Share of position-weeks where ties make the group longer than K:")
display(
    group_size.assign(longer=group_size["n_true"] > group_size["k"])
    .groupby("position")["longer"]
    .mean()
    .reindex(list(K_BY_POSITION))
)
print("players seen in 2+ seasons:", int(panel.groupby("code")["season"].nunique().ge(2).sum()))
gw1 = panel[(panel["season"] == "2025-26") & (panel["GW"] == 1)]
print("2025-26 GW1 with prior minutes:", float(gw1["minutes_ewm"].notna().mean()))
print("rows with past band 0:", float((panel["team_past_band"] == 0).mean()))
print("max home matches in a player-week:", int(panel["n_home"].max()))
print("attacking_share missing rate:")
display(panel.groupby("season")["attacking_share"].apply(lambda s: float(s.isna().mean())))
display(panel[["name", "season", "GW", "team_past_band", "team_current_band", "opp_past_band_hard", "opp_past_band_easy", "n_home", "minutes_ewm"]].head())
display(test_df[["name", "position", "GW", "minutes_ewm", "team_current_band", "opp_current_band_hard", "opp_current_band_easy", "n_home"]].head())


## Model


Four jobs, one per position. The training answer is 1 when the player's tie-aware rank is inside that position's K, and 0 otherwise.

Logistic regression and gradient boosting each predict the chance of a 1. Each gameweek has equal total weight, and each player inside it has equal weight. Recent minutes is a column, not a weight.

The choice uses 2025-26 GW20 through GW38. Each week is fit on earlier rows only. A fitted model is kept only when its average hit rate beats sorting by recent minutes. A simpler model wins a tie between the two fits.

This season's finished weeks replay that frozen choice. They do not change the model or the columns. The next-gameweek list is a new fit of the frozen choice on every finished row.

The position-specific challenger has three time layers. For every scored week, training contains only earlier rows.

1. **Development and model choice:** 2025-26 GW1-19 compares logistic regression, gradient boosting, and recent-minutes sorting.
2. **Historical check:** the chosen challenger is replayed on 2025-26 GW20-38. This is useful evidence, but not a clean test because those weeks informed earlier feature discussions.
3. **Prospective test:** 2026-27 GW11-16 is untouched. Challenger columns and selected model family stay frozen throughout this block. Its lists are shadow lists until GW16 finishes.


In [ ]:
position_order = list(K_BY_POSITION)

base_feature_columns = [
    "minutes_ewm",
    "zero_min_rate_5",
    "start_rate_5",
    "minutes_60_rate",
    "xg_per90",
    "xa_per90",
    "saves_per90",
    "def_actions_per90",
    "xg_form",
    "xa_form",
    "minutes_form",
    "team_xg_per_match_5",
    "team_xga_per_match_5",
    "xg_matchup",
    "xa_matchup",
    "concede_matchup",
    "n_fixtures",
    "n_home",
    "team_past_band",
    "team_current_band",
    "opp_past_band_hard",
    "opp_past_band_easy",
    "opp_current_band_hard",
    "opp_current_band_easy",
    "value",
]
# Shared list. Midfield adds the attacking share.
feature_columns = {
    position: base_feature_columns.copy() for position in position_order
}
feature_columns["MID"] = feature_columns["MID"] + ["attacking_share"]

# Fixed shadow challenger. These lists are not selected on the choice weeks.
challenger_feature_columns = {
    "GK": [
        "minutes_ewm", "zero_min_rate_5", "start_rate_5", "minutes_60_rate",
        "minutes_form", "saves_per90_5", "bps_per90_5", "bonus_rate_5",
        "team_xga_venue_5", "opp_xg_venue_5", "venue_defense_matchup",
        "team_clean_sheet_venue_rate_5", "n_fixtures", "n_home", "value",
    ],
    "DEF": [
        "minutes_ewm", "zero_min_rate_5", "start_rate_5", "minutes_60_rate",
        "minutes_form", "xg_per90_5", "xa_per90_5", "xg_per90_10",
        "xa_per90_10", "threat_per90_5", "creativity_per90_5",
        "def_contribution_rate_5", "bps_per90_5", "bonus_rate_5",
        "clean_sheet_rate_5", "team_xga_venue_5", "opp_xg_venue_5",
        "venue_defense_matchup", "team_clean_sheet_venue_rate_5",
        "n_fixtures", "n_home", "value",
    ],
    "MID": [
        "minutes_ewm", "zero_min_rate_5", "start_rate_5", "minutes_60_rate",
        "minutes_form", "xg_per90_5", "xa_per90_5", "xg_per90_10",
        "xa_per90_10", "goals_per90_10", "assists_per90_10",
        "threat_per90_5", "creativity_per90_5", "xgi_team_rate_ratio_5",
        "def_contribution_rate_5", "bps_per90_5", "bonus_rate_5",
        "team_xg_venue_5", "opp_xga_venue_5", "venue_attack_matchup",
        "n_fixtures", "n_home", "value",
    ],
    "FWD": [
        "minutes_ewm", "zero_min_rate_5", "start_rate_5", "minutes_60_rate",
        "minutes_form", "xg_per90_5", "xa_per90_5", "xg_per90_10",
        "goals_per90_10", "assists_per90_10", "threat_per90_5",
        "xgi_team_rate_ratio_5", "bps_per90_5", "bonus_rate_5",
        "team_xg_venue_5", "opp_xga_venue_5", "venue_attack_matchup",
        "n_fixtures", "n_home", "value",
    ],
}
challenger_choice_origins = [("2025-26", gw) for gw in range(1, 20)]
challenger_check_origins = [("2025-26", gw) for gw in range(20, 39)]
challenger_start_gw = 11
challenger_end_gw = 16

models_to_grade = ["logistic_regression", "gradient_boosting"]
reference_name = "minutes_sort"
model_simplicity = {
    "logistic_regression": 0,
    "gradient_boosting": 1,
}


def top_ids_with_ties(group, column, k):
    """IDs at or above the K-th value. Ties on the cutoff are all included."""
    values = group[column]
    ranked = values.dropna().sort_values(ascending=False)
    if ranked.empty:
        return set()
    k = min(k, len(ranked))
    cutoff = ranked.iloc[k - 1]
    return set(group.loc[values >= cutoff, id_column])


def precision_at_k_detail(group, score_column, points_column, k):
    top_pred = top_ids_with_ties(group, score_column, k)
    top_true = top_ids_with_ties(group, points_column, k)
    overlap = len(top_pred & top_true)
    precision = overlap / len(top_pred) if top_pred else np.nan
    recall = overlap / len(top_true) if top_true else np.nan
    return {
        "precision_at_k": precision,
        "recall_at_k": recall,
        "n_pred": len(top_pred),
        "n_true": len(top_true),
        "k": k,
    }


def precision_at_k(data, score_column, points_column=target_column):
    """Mean tie-aware Precision@K over gameweeks. `data` is one position."""
    position = data["position"].iloc[0]
    k = K_BY_POSITION[position]
    scores = []
    for _, group in data.groupby(time_column):
        detail = precision_at_k_detail(group, score_column, points_column, k)
        scores.append(detail["precision_at_k"])
    return float(np.nanmean(scores)) if scores else np.nan


def build_X(data, columns, reference_columns=None):
    X = data[columns].copy()
    if reference_columns is None:
        return X
    return X.reindex(columns=reference_columns, fill_value=0)


def rows_before(data, season, gw):
    order = data["season"].map(SEASON_ORDER)
    target = SEASON_ORDER[season]
    return (order < target) | ((data["season"] == season) & (data[time_column] < gw))


def frame_precision(frame, scores):
    scored = frame[[time_column, "position", id_column, target_column]].copy()
    scored["score"] = np.asarray(scores)
    return precision_at_k(scored, "score")


def week_weight(data):
    """Each gameweek sums to 1. Each player inside it has equal weight."""
    week_n = data.groupby(["season", time_column])[id_column].transform("size")
    return (1 / week_n).to_numpy()


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

logistic_kwargs = {"max_iter": 1000, "random_state": random_state}
boosting_kwargs = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 3,
    "random_state": random_state,
}


def make_model(model_name):
    if model_name == "logistic_regression":
        return LogisticRegression(**logistic_kwargs)
    if model_name == "gradient_boosting":
        return GradientBoostingClassifier(**boosting_kwargs)
    return None


def fit_scores(train_part, predict_parts, model_name, columns_by_position=None):
    """Fit on train_part. Return scores for each frame in predict_parts, then the model."""
    if model_name == reference_name:
        scores = [part["minutes_ewm"].fillna(-1).to_numpy() for part in predict_parts]
        return scores, None

    if columns_by_position is None:
        columns_by_position = feature_columns
    position = train_part["position"].iloc[0]
    columns = columns_by_position[position]
    X_train = build_X(train_part, columns)
    X_predict = [
        build_X(part, columns, reference_columns=X_train.columns)
        for part in predict_parts
    ]
    y_train = train_part["y"].to_numpy()
    weight = week_weight(train_part)

    imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    train_i = imputer.fit_transform(X_train)
    predict_i = [imputer.transform(part) for part in X_predict]
    if model_name == "logistic_regression":
        scaler = StandardScaler()
        train_i = scaler.fit_transform(train_i)
        predict_i = [scaler.transform(part) for part in predict_i]

    model = make_model(model_name)
    model.fit(train_i, y_train, sample_weight=weight)
    scores = [model.predict_proba(part)[:, 1] for part in predict_i]
    return scores, model


def recent_three_weeks(train_part):
    recent_keys = (
        train_part[["season", "season_order", time_column]]
        .drop_duplicates()
        .sort_values(["season_order", time_column])
        .tail(3)
    )
    recent_mask = pd.Series(False, index=train_part.index)
    for recent_season, _, recent_gw in recent_keys.itertuples(index=False):
        recent_mask = recent_mask | (
            (train_part["season"] == recent_season)
            & (train_part[time_column] == recent_gw)
        )
    return train_part.loc[recent_mask]


def score_origin(position_rows, model_name, season, gw, position):
    train_part = position_rows.loc[rows_before(position_rows, season, gw)]
    valid_part = position_rows[
        (position_rows["season"] == season) & (position_rows[time_column] == gw)
    ]
    if train_part.empty or valid_part.empty or train_part["y"].nunique() < 2:
        return None
    recent_part = recent_three_weeks(train_part)
    (recent_scores, valid_scores), _ = fit_scores(
        train_part, [recent_part, valid_part], model_name
    )
    return {
        "position": position,
        "model": model_name,
        "season": season,
        "GW": gw,
        "train_precision_at_k": frame_precision(recent_part, recent_scores),
        "val_precision_at_k": frame_precision(valid_part, valid_scores),
    }


labeled = panel[panel[target_column].notna()].copy()
labeled = add_top_group_target(labeled)

choice_origins = [("2025-26", gw) for gw in range(20, 39)]
grade_rows = []
for position in position_order:
    position_rows = labeled[labeled["position"] == position].copy()
    for model_name in models_to_grade + [reference_name]:
        for season, gw in choice_origins:
            row = score_origin(position_rows, model_name, season, gw, position)
            if row is not None:
                grade_rows.append(row)

grades_df = pd.DataFrame(grade_rows)
results_df = (
    grades_df.groupby(["position", "model"], as_index=False)
    .agg(
        val_precision_at_k=("val_precision_at_k", "mean"),
        train_precision_at_k=("train_precision_at_k", "mean"),
    )
)
sort_precision = results_df.loc[
    results_df["model"] == reference_name,
    ["position", "val_precision_at_k"],
].rename(columns={"val_precision_at_k": "minutes_sort_precision_at_k"})
results_df = results_df.merge(sort_precision, on="position", how="left")
results_df["lift"] = (
    results_df["val_precision_at_k"] - results_df["minutes_sort_precision_at_k"]
)
results_df["simplicity"] = results_df["model"].map(model_simplicity)
results_df["position"] = pd.Categorical(
    results_df["position"], position_order, ordered=True
)
results_df = results_df.sort_values(
    ["position", "val_precision_at_k", "simplicity"],
    ascending=[True, False, True],
)
display(results_df.drop(columns="simplicity"))

chosen_rows = []
fitted = results_df[results_df["model"] != reference_name]
for position, group in fitted.groupby("position", sort=False, observed=True):
    best = group.iloc[0]
    sort_hit = best["minutes_sort_precision_at_k"]
    if best["val_precision_at_k"] > sort_hit:
        chosen_rows.append(best)
    else:
        chosen_rows.append(
            results_df[
                (results_df["position"] == position)
                & (results_df["model"] == reference_name)
            ].iloc[0]
        )
chosen_df = pd.DataFrame(chosen_rows).reset_index(drop=True)
print("Chosen model per position. A fit is kept only when it beats the minutes sort.")
display(
    chosen_df[
        [
            "position",
            "model",
            "val_precision_at_k",
            "train_precision_at_k",
            "minutes_sort_precision_at_k",
            "lift",
        ]
    ]
)


In [ ]:
# Frozen choice. This season's weeks are a replay, not a new model choice.
replay_origins = [(current_season, int(gw)) for gw in finished_gws]
replay_rows = []
for position, model_name in chosen_df.set_index("position")["model"].items():
    position_rows = labeled[labeled["position"] == position].copy()
    for season, gw in replay_origins:
        row = score_origin(position_rows, model_name, season, gw, position)
        if row is not None:
            replay_rows.append(row)

replay_df = pd.DataFrame(replay_rows)
print("Replay hit rate by position, frozen model, this season's finished weeks:")
display(
    replay_df.groupby("position")["val_precision_at_k"]
    .mean()
    .reindex(position_order)
)
display(
    replay_df.pivot(index="GW", columns="position", values="val_precision_at_k")
    .reindex(columns=position_order)
)

# Live list: a new fit per position on every finished row of that position.
test_pred_parts = []
live_models = {}
for position, model_name in chosen_df.set_index("position")["model"].items():
    position_rows = labeled[labeled["position"] == position].copy()
    position_test = test_df[test_df["position"] == position].copy()
    live_scores, live_model = fit_scores(position_rows, [position_test], model_name)
    part = position_test.copy()
    part["score"] = live_scores[0]
    part["model"] = model_name
    test_pred_parts.append(part)
    live_models[position] = live_model
    print(
        f"{position}: {model_name}, live fit on {len(position_rows)} rows, predicting GW {test_gw}"
    )

test_pred_df = pd.concat(test_pred_parts, ignore_index=True)


### Position-specific shadow challenger

The frozen model remains the official list. The challenger compares logistic regression, gradient boosting, and recent-minutes sorting on 2025-26 GW1-19, separately by position. That choice is replayed on GW20-38 for a historical check. It then starts the untouched GW11-16 shadow test without changing columns or model family. Every scored week is fit only on earlier rows. The live shadow list is printed for comparison, but it does not replace the official list during this block.


In [ ]:
def score_challenger_origin(position_rows, model_name, season, gw, position):
    train_part = position_rows.loc[rows_before(position_rows, season, gw)]
    valid_part = position_rows[
        (position_rows["season"] == season) & (position_rows[time_column] == gw)
    ]
    if train_part.empty or valid_part.empty or train_part["y"].nunique() < 2:
        return None
    valid_scores, _ = fit_scores(
        train_part,
        [valid_part],
        model_name,
        columns_by_position=challenger_feature_columns,
    )
    return {
        "position": position,
        "model": model_name,
        "season": season,
        "GW": gw,
        "val_precision_at_k": frame_precision(valid_part, valid_scores[0]),
    }


# Development: choose one challenger model per position on 2025-26 GW1-19.
challenger_grade_rows = []
for position in position_order:
    position_rows = labeled[labeled["position"] == position].copy()
    for model_name in models_to_grade + [reference_name]:
        for season, gw in challenger_choice_origins:
            row = score_challenger_origin(
                position_rows, model_name, season, gw, position
            )
            if row is not None:
                challenger_grade_rows.append(row)

challenger_grades_df = pd.DataFrame(challenger_grade_rows)
challenger_results_df = (
    challenger_grades_df.groupby(["position", "model"], as_index=False)
    .agg(
        choice_hit_rate=("val_precision_at_k", "mean"),
        n_choice_weeks=("GW", "nunique"),
    )
)
challenger_sort = challenger_results_df[
    challenger_results_df["model"] == reference_name
][["position", "choice_hit_rate"]].rename(
    columns={"choice_hit_rate": "minutes_sort_hit_rate"}
)
challenger_results_df = challenger_results_df.merge(
    challenger_sort, on="position", how="left"
)
challenger_results_df["lift"] = (
    challenger_results_df["choice_hit_rate"]
    - challenger_results_df["minutes_sort_hit_rate"]
)
challenger_results_df["simplicity"] = challenger_results_df["model"].map(
    model_simplicity
)
challenger_results_df["position"] = pd.Categorical(
    challenger_results_df["position"], position_order, ordered=True
)
challenger_results_df = challenger_results_df.sort_values(
    ["position", "choice_hit_rate", "simplicity"],
    ascending=[True, False, True],
)
print("Challenger model choice, 2025-26 GW1-19:")
display(challenger_results_df.drop(columns="simplicity"))

challenger_chosen_rows = []
challenger_fitted = challenger_results_df[
    challenger_results_df["model"] != reference_name
]
for position, group in challenger_fitted.groupby(
    "position", sort=False, observed=True
):
    best = group.iloc[0]
    sort_hit = best["minutes_sort_hit_rate"]
    if best["choice_hit_rate"] > sort_hit:
        challenger_chosen_rows.append(best)
    else:
        challenger_chosen_rows.append(
            challenger_results_df[
                (challenger_results_df["position"] == position)
                & (challenger_results_df["model"] == reference_name)
            ].iloc[0]
        )
challenger_chosen_df = pd.DataFrame(challenger_chosen_rows).reset_index(drop=True)
challenger_model_by_position = challenger_chosen_df.set_index("position")["model"].to_dict()
print("Chosen challenger per position:")
display(
    challenger_chosen_df[[
        "position", "model", "choice_hit_rate", "minutes_sort_hit_rate", "lift",
    ]]
)

# Historical check: replay the chosen challenger on GW20-38 without re-selecting it.
challenger_check_rows = []
for position, model_name in challenger_model_by_position.items():
    position_rows = labeled[labeled["position"] == position].copy()
    for season, gw in challenger_check_origins:
        row = score_challenger_origin(
            position_rows, model_name, season, gw, position
        )
        if row is not None:
            challenger_check_rows.append(row)
challenger_check_df = pd.DataFrame(challenger_check_rows)
challenger_check_summary = (
    challenger_check_df.groupby("position", as_index=False)
    .agg(
        challenger_check_hit_rate=("val_precision_at_k", "mean"),
        n_check_weeks=("GW", "nunique"),
    )
)
official_check = chosen_df[["position", "model", "val_precision_at_k"]].rename(
    columns={
        "model": "official_model",
        "val_precision_at_k": "official_check_hit_rate",
    }
)
challenger_check_summary = challenger_check_summary.merge(
    official_check, on="position", how="left"
)
challenger_check_summary["difference"] = (
    challenger_check_summary["challenger_check_hit_rate"]
    - challenger_check_summary["official_check_hit_rate"]
)
print("Historical check, 2025-26 GW20-38 (not a clean final test):")
display(challenger_check_summary.set_index("position").reindex(position_order))


shadow_finished_gws = [
    int(gw) for gw in finished_gws
    if challenger_start_gw <= int(gw) <= challenger_end_gw
]
challenger_replay_rows = []
for position, model_name in challenger_model_by_position.items():
    position_rows = labeled[labeled["position"] == position].copy()
    for gw in shadow_finished_gws:
        row = score_challenger_origin(
            position_rows, model_name, current_season, gw, position
        )
        if row is not None:
            challenger_replay_rows.append(row)

challenger_replay_df = pd.DataFrame(challenger_replay_rows)
if challenger_replay_df.empty:
    print(
        f"Shadow evaluation waits for GW{challenger_start_gw}. "
        f"The fixed block is GW{challenger_start_gw}-{challenger_end_gw}."
    )
else:
    official_shadow = replay_df[
        replay_df[time_column].between(challenger_start_gw, challenger_end_gw)
    ].copy()
    official_summary = (
        official_shadow.groupby("position")["val_precision_at_k"]
        .mean()
        .rename("official_hit_rate")
    )
    challenger_summary = (
        challenger_replay_df.groupby("position")["val_precision_at_k"]
        .mean()
        .rename("challenger_hit_rate")
    )
    shadow_summary = pd.concat([official_summary, challenger_summary], axis=1)
    shadow_summary["difference"] = (
        shadow_summary["challenger_hit_rate"] - shadow_summary["official_hit_rate"]
    )
    print(
        f"Completed shadow weeks: {shadow_finished_gws}. "
        f"Do not choose a winner before GW{challenger_end_gw} is finished."
    )
    display(shadow_summary.reindex(position_order))
    display(
        challenger_replay_df.pivot(
            index="GW", columns="position", values="val_precision_at_k"
        ).reindex(columns=position_order)
    )

# Current shadow list. It is for comparison only during GW11-16.
challenger_test_parts = []
challenger_live_models = {}
for position in position_order:
    position_rows = labeled[labeled["position"] == position].copy()
    position_test = test_df[test_df["position"] == position].copy()
    model_name = challenger_model_by_position[position]
    shadow_scores, shadow_model = fit_scores(
        position_rows,
        [position_test],
        model_name,
        columns_by_position=challenger_feature_columns,
    )
    part = position_test.copy()
    part["score"] = shadow_scores[0]
    part["model"] = model_name
    challenger_test_parts.append(part)
    challenger_live_models[position] = shadow_model

challenger_test_pred_df = pd.concat(challenger_test_parts, ignore_index=True)
challenger_top_lists = {}
for position, k in K_BY_POSITION.items():
    group = challenger_test_pred_df[
        challenger_test_pred_df["position"] == position
    ]
    top_ids = top_ids_with_ties(group, "score", k)
    challenger_top_lists[position] = (
        group[group[id_column].isin(top_ids)]
        [[id_column, "name", "team", "position", "score", "value", "model"]]
        .sort_values(["score", "name"], ascending=[False, True])
        .reset_index(drop=True)
    )
    print(
        f"\n{position} shadow: {challenger_model_by_position[position]}, target K={k}, "
        f"shortlist size={len(challenger_top_lists[position])} "
        f"(test GW {test_gw}, comparison only)"
    )
    display(challenger_top_lists[position])


In [ ]:
print("Choice weeks, 2025-26 GW20-38. Previous 3 weeks are already inside the fit:")
display(
    chosen_df[
        [
            "position",
            "model",
            "train_precision_at_k",
            "val_precision_at_k",
            "minutes_sort_precision_at_k",
        ]
    ]
)
print("Replay weeks, frozen model:")
display(
    replay_df.pivot(index="GW", columns="position", values="val_precision_at_k")
    .reindex(columns=position_order)
)


In [ ]:
# test GW shortlists from the live fit. Ties at the cutoff stay in the list.
top_lists = {}
chosen_model = chosen_df.set_index("position")["model"]
for position, k in K_BY_POSITION.items():
    group = test_pred_df[test_pred_df["position"] == position]
    top_ids = top_ids_with_ties(group, "score", k)
    top_lists[position] = (
        group[group[id_column].isin(top_ids)]
        [[id_column, "name", "team", "position", "score", "value", "model"]]
        .sort_values(["score", "name"], ascending=[False, True])
        .reset_index(drop=True)
    )
    print(
        f"\n{position}: {chosen_model[position]}, target K={k}, "
        f"shortlist size={len(top_lists[position])} "
        f"(ties included, test GW {test_gw}, live fit)"
    )
    display(top_lists[position])


### Plots

The bars are the 2025-26 choice. A fitted model is kept only when its bar beats the minutes sort. The line is the frozen replay on this season's finished weeks. The last chart is the live shortlist.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
model_order = models_to_grade + [reference_name]
plot_results = results_df.copy()
plot_results["model"] = pd.Categorical(plot_results["model"], model_order, ordered=True)

fig, ax = plt.subplots(figsize=(8, 3.8))
sns.barplot(
    data=plot_results,
    x="position",
    y="val_precision_at_k",
    hue="model",
    hue_order=model_order,
    order=position_order,
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.set_ylabel("Average next-week hit rate")
ax.set_title("2025-26 GW20-38 choice")
ax.legend(title="")
plt.tight_layout()
plt.show()

replay_plot = replay_df.copy()
fig, ax = plt.subplots(figsize=(8, 3.8))
sns.lineplot(
    data=replay_plot,
    x="GW",
    y="val_precision_at_k",
    hue="position",
    hue_order=position_order,
    marker="o",
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_xlabel("2026-27 gameweek")
ax.set_ylabel("Hit rate")
ax.set_title("Frozen replay")
ax.legend(title="")
plt.tight_layout()
plt.show()

test_plot_df = pd.concat(top_lists.values(), ignore_index=True)
top_plot = (
    test_plot_df.sort_values("score", ascending=False)
    .groupby("position", group_keys=False)
    .head(10)
)
g = sns.catplot(
    data=top_plot,
    y="name",
    x="score",
    col="position",
    kind="bar",
    color="#55A868",
    sharey=False,
    height=4.5,
    aspect=0.7,
    col_order=position_order,
)
g.set_axis_labels("", "")
for position, ax in zip(position_order, g.axes.flat):
    if position == "GK":
        ax.set_xlabel("Recent minutes")
    else:
        ax.set_xlabel("Chance of the top group")
g.figure.suptitle(f"Test GW {test_gw} shortlists (ties included)", y=1.03)
plt.show()
